# lyrics Text Analysis (TF-IDF, LDA)

Yangshunshun (Luna) Lu

The file did textual analysis modeling for lyrics using TF-IDF and LDA

No AI used

## TF-IDF

In [8]:
import pandas as pd
import os
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from textblob import TextBlob

In [9]:
# ================= 1. TF-IDF =================
base_file = os.path.join('..', 'Data', 'cleaned_data', 'tokenized_lyrics_data.csv')
model_file = os.path.join('..', 'Data', 'cleaned_data', 'modeling_results.csv')

if os.path.exists(model_file):
    df = pd.read_csv(model_file)
else:
    df = pd.read_csv(base_file)

df['processed_lyrics'] = df['processed_lyrics'].fillna('')
docs = df['processed_lyrics'].tolist()

tfidf_vectorizer = TfidfVectorizer(max_df=0.9, min_df=5)
tfidf_matrix = tfidf_vectorizer.fit_transform(docs)
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

# --- Output A: Save global TF-IDF ranking table (independent file) ---
sums = tfidf_matrix.sum(axis=0)
data = [(term, sums[0, col]) for col, term in enumerate(tfidf_feature_names)]
global_tfidf_df = pd.DataFrame(data, columns=['term', 'TF-IDF_score']).sort_values('TF-IDF_score', ascending=False)
global_output = os.path.join('..', 'Data', 'cleaned_data', 'global_tfidf_ranking.csv')
global_tfidf_df.to_csv(global_output, index=False, encoding='utf-8-sig')

# --- Output B: Generate Top 3 exclusive keywords per song (written to model file) ---
def get_top_tf_idf_words(row_idx, top_n=3):
    row_vector = tfidf_matrix.getrow(row_idx).toarray()[0]
    top_indices = row_vector.argsort()[-top_n:][::-1]
    top_words = [tfidf_feature_names[i] for i in top_indices if row_vector[i] > 0]
    return ", ".join(top_words)

df['Top_Keywords'] = [get_top_tf_idf_words(i) for i in range(tfidf_matrix.shape[0])]

df.to_csv(model_file, index=False, encoding='utf-8-sig')

In [10]:
# ================= 2. LDA Topic Modeling Module =================
base_file = os.path.join('..', 'Data', 'cleaned_data', 'tokenized_lyrics_data.csv')
model_file = os.path.join('..', 'Data', 'cleaned_data', 'modeling_results.csv')

if os.path.exists(model_file):
    df = pd.read_csv(model_file)
else:
    df = pd.read_csv(base_file)

df['processed_lyrics'] = df['processed_lyrics'].fillna('')
docs = df['processed_lyrics'].tolist()

tf_vectorizer = CountVectorizer(max_df=0.9, min_df=5)
tf_matrix = tf_vectorizer.fit_transform(docs)
tf_feature_names = tf_vectorizer.get_feature_names_out()

N_TOPICS = 5
lda_model = LatentDirichletAllocation(n_components=N_TOPICS, max_iter=10, random_state=42)
lda_matrix = lda_model.fit_transform(tf_matrix)

for topic_idx, topic in enumerate(lda_model.components_):
    top_words = [tf_feature_names[i] for i in topic.argsort()[:-9:-1]]
    print(f"topic {topic_idx}: {', '.join(top_words)}")

existing_topic_cols = [col for col in df.columns if 'Topic_' in col or 'Dominant_Topic' in col]
if existing_topic_cols:
    df = df.drop(columns=existing_topic_cols)

topic_cols = [f"Topic_{i}_Prob" for i in range(N_TOPICS)]
lda_df = pd.DataFrame(lda_matrix, columns=topic_cols)
lda_df['Dominant_Topic'] = np.argmax(lda_matrix, axis=1)

df = pd.concat([df, lda_df], axis=1)

if 'word_count' not in df.columns:
    df['word_count'] = df['processed_lyrics'].apply(lambda x: len(str(x).split()))

df.to_csv(model_file, index=False, encoding='utf-8-sig')
print(f"Completed! LDA saved to: {model_file}")

topic 0: si, quier, hac, to, sol, dic, baby, call
topic 1: si, quier, sol, amor, se, siempr, voy, vid
topic 2: na, quier, si, noch, pas, pued, mir, hac
topic 3: î¼î, î½î, î¹, î½, îºî, îµï, îµî, îµ
topic 4: va, pong, mal, το, ta, μου, si, να
Completed! LDA saved to: ..\Data\cleaned_data\modeling_results.csv


In [11]:

model_file = os.path.join('..', 'Data', 'cleaned_data', 'modeling_results.csv')
df = pd.read_csv(model_file)
df['processed_lyrics'] = df['processed_lyrics'].fillna('')


def calculate_nlp_metrics(text):
    words = str(text).split()
    word_count = len(words)
    if word_count == 0:
        return 0, 0, 0
    
    unique_words = len(set(words))
    lexical_richness = unique_words / word_count
    
    total_chars = sum(len(word) for word in words)
    avg_word_length = total_chars / word_count
    
    return word_count, lexical_richness, avg_word_length

metrics = df['processed_lyrics'].apply(calculate_nlp_metrics)
df['word_count'] = [m[0] for m in metrics]
df['lexical_richness'] = [m[1] for m in metrics]
df['avg_word_length'] = [m[2] for m in metrics]

df.to_csv(model_file, index=False, encoding='utf-8-sig')